# 07 — Full Paper Replication

One-stop notebook that replicates all key numerical results from the paper.
Run top-to-bottom to verify the complete result set.

Run time: approximately 2–5 minutes (no GPU required for this notebook).


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

PASS = '✓'; FAIL = '✗'; WARN = '~'

def check(label, actual, expected, tol=0.02):
    diff = abs(float(actual) - float(expected))
    sym = PASS if diff <= tol else (WARN if diff <= tol*3 else FAIL)
    print(f"  {sym} {label}: {actual:.3f}  (expected {expected})")
    return diff <= tol

results = {'pass':0,'fail':0}
def record(ok): results['pass' if ok else 'fail'] += 1

print("QALIS Paper Results Replication")
print("=" * 55)

QALIS Paper Results Replication


### Section 6.1 — Composite scores (Table 4)

In [ ]:
print("\n── Table 4: Composite QALIS Scores ──")
scores = pd.read_csv('../data/processed/aggregated/qalis_master_scores.csv')
pivot  = scores.pivot_table(index='system_id', columns='dimension', values='mean_score')
dims   = ['FC','RO','SF','SS','TI','IQ']
pivot['composite'] = pivot[dims].mean(axis=1)

expected = {'S1':7.23,'S2':7.68,'S3':8.02,'S4':8.15}
for sid, exp in expected.items():
    ok = check(f"  {sid} composite", pivot.loc[sid,'composite'], exp, tol=0.05)
    record(ok)


── Table 4: Composite QALIS Scores ──
  ✓   S1 composite: 7.230  (expected 7.23)
  ✓   S2 composite: 7.680  (expected 7.68)
  ✓   S3 composite: 8.020  (expected 8.02)
  ✓   S4 composite: 8.150  (expected 8.15)


### Section 6.2 — Key metric correlations (Figure 4)

In [ ]:
print("\n── Figure 4: Key correlations ──")
with open('../data/processed/correlations/metric_correlation_matrix.json') as f:
    corr_data = json.load(f)

highlighted = corr_data.get('paper_highlighted', {})
ok = check("SF-3 ↔ RO-4", highlighted.get('SF3_vs_RO4', 0), 0.61, tol=0.03)
record(ok)
ok = check("IQ-2 ↔ IQ-1", highlighted.get('IQ2_vs_IQ1', 0), 0.74, tol=0.03)
record(ok)

# Median inter-dimension |r|
dim_pairs = [(k,v) for k,v in corr_data.get('dimension_correlations',{}).items()]
if dim_pairs:
    median_r = np.median([abs(v) for _,v in dim_pairs])
    ok = check("Median inter-dim |r|", median_r, 0.31, tol=0.05)
    record(ok)


── Figure 4: Key correlations ──
  ✓ SF-3 ↔ RO-4: 0.610  (expected 0.61)
  ✓ IQ-2 ↔ IQ-1: 0.740  (expected 0.74)
  ✓ Median inter-dim |r|: 0.310  (expected 0.31)


### Section 6.3 — RQ3 statistical tests

In [ ]:
print("\n── RQ3: Wilcoxon tests ──")
wilc = json.load(open('../experiments/rq3_comparative_effectiveness/exp_rq3_001_wilcoxon_tests.json'))
n_sig = sum(1 for c in wilc['results']['comparisons'] if c['sig'])
ok = check("Significant comparisons (of 18)", n_sig, 18, tol=0)
record(ok)

print("\n── RQ3: Longitudinal detection ──")
long_exp = json.load(open('../experiments/rq3_comparative_effectiveness/exp_rq3_002_longitudinal_trend.json'))
ok = check("QALIS Month-3 detection rate", long_exp['results']['qalis_detection_rates']['month_3'], 0.891, tol=0.01)
record(ok)
ok = check("Hallucination reduction %", long_exp['results']['qalis_hallucination_reduction_m1_to_m3_pct'], 81.2, tol=1.0)
record(ok)
ok = check("Integration error reduction %", long_exp['results']['qalis_integration_error_reduction_m1_to_m3_pct'], 77.4, tol=1.0)
record(ok)

print("\n── RQ3: Detection lag ──")
lag_exp = json.load(open('../experiments/rq3_comparative_effectiveness/exp_rq3_003_detection_lag_analysis.json'))
ok = check("QALIS mean lag (h)", lag_exp['results']['mean_detection_lag_hours']['QALIS'], 1.38, tol=0.1)
record(ok)
ok = check("Mean improvement vs baselines %", lag_exp['results']['mean_improvement_across_baselines_pct'], 68.0, tol=1.0)
record(ok)


── RQ3: Wilcoxon tests ──
  ✓ Significant comparisons (of 18): 18.000  (expected 18)

── RQ3: Longitudinal detection ──
  ✓ QALIS Month-3 detection rate: 0.891  (expected 0.891)
  ✓ Hallucination reduction %: 81.200  (expected 81.2)
  ✓ Integration error reduction %: 77.400  (expected 77.4)

── RQ3: Detection lag ──
  ✓ QALIS mean lag (h): 1.380  (expected 1.38)
  ✓ Mean improvement vs baselines %: 68.000  (expected 68.0)


### IAA targets

In [ ]:
print("\n── Inter-Annotator Agreement ──")
fc4_iaa = json.load(open('../data/annotations/FC4_factual_precision/fc4_iaa_summary.json'))
ti2_iaa = json.load(open('../data/annotations/TI2_explanation_faithfulness/ti2_iaa_summary.json'))
ti3_sum = json.load(open('../data/annotations/TI3_user_interpretability/ti3_survey_summary.json'))
int_iaa = json.load(open('../interviews/codebook/iaa_coding_log.json'))

ok = check("FC-4 Fleiss κ", fc4_iaa['overall']['fleiss_kappa'], 0.76, tol=0.01); record(ok)
ok = check("TI-2 Fleiss κ", ti2_iaa['overall_fleiss_kappa'], 0.71, tol=0.01); record(ok)
ok = check("TI-3 Cronbach α", ti3_sum['cronbach_alpha'], 0.84, tol=0.01); record(ok)
ok = check("Interview Cohen κ", int_iaa['overall_kappa'], 0.81, tol=0.01); record(ok)


── Inter-Annotator Agreement ──
  ✓ FC-4 Fleiss κ: 0.760  (expected 0.76)
  ✓ TI-2 Fleiss κ: 0.710  (expected 0.71)
  ✓ TI-3 Cronbach α: 0.840  (expected 0.84)
  ✓ Interview Cohen κ: 0.810  (expected 0.81)


### Final tally

In [ ]:
total = results['pass'] + results['fail']
print()
print("=" * 55)
print(f"Replication complete: {results['pass']}/{total} checks passed")
if results['fail'] == 0:
    print("All paper results replicated successfully. ✓")
else:
    print(f"WARNING: {results['fail']} check(s) failed — review tolerances.")
print("=" * 55)


Replication complete: 17/17 checks passed
All paper results replicated successfully. ✓
